# Heritage Short: GPU runner (Kaggle)

Runs the bundles made by `film bundle shots` and `film bundle music` (ADR-006). The Wan 2.2 clips run on GPU 0 and the ACE-Step candidates on GPU 1, at the same time.

1. In this notebook: **Settings → Accelerator → GPU T4 x2**, and **Internet → on** (this needs a phone-verified account).
2. **Add Input** → your dataset with the bundle zips (Kaggle unpacks them).
3. Set `LIMIT` below: **1 for spike S1**, which runs a single job to measure time and memory, or 0 for everything.
4. **Save Version → Save & Run All** so it keeps running after you close the tab.
5. When it finishes, download `heritage-gpu-out.zip` from the Output tab, then in the repo run `uv run film ingest heritage-gpu-out.zip`.

Every job writes its output as soon as it is done, plus one line in `results.jsonl` (failures included), so a session that dies loses only the job it was on.

In [ ]:
LIMIT = 1          # spike S1/S2: one job per bundle. Set 0 to run every job.
HF_HOME = "/tmp/hf" # Wan's weights are about 34 GB: keep them on scratch, not in /kaggle/working (20 GB, saved)

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

n = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(n)])
assert n >= 1, "No GPU: Settings → Accelerator → GPU T4 x2"
for i in range(n):
    assert torch.cuda.get_device_capability(i) >= (7, 5), "This GPU is too old for current PyTorch; choose GPU T4 x2"
for d in ("/tmp", "/kaggle/working"):
    u = shutil.disk_usage(d); print(f"{d}: {u.free / 2**30:.0f} GB free")

In [ ]:
!pip install -q ftfy imageio imageio-ffmpeg uv

In [ ]:
# Find the bundles: folders holding a manifest.json (Kaggle unpacks uploaded zips), or zips it didn't unpack.
staging = Path("/tmp/bundles"); staging.mkdir(exist_ok=True)
for z in Path("/kaggle/input").rglob("*.zip"):
    with zipfile.ZipFile(z) as f:
        if "manifest.json" in f.namelist():
            f.extractall(staging / z.stem)
bundles = sorted({p.parent for p in list(Path("/kaggle/input").rglob("manifest.json")) + list(staging.rglob("manifest.json"))})
for b in bundles:
    m = json.loads((b / "manifest.json").read_text())
    print(f"{m['bundle']}: {m['stage']}, {len(m['jobs'])} jobs -> {[j['id'] for j in m['jobs']]}")
assert bundles, "No bundle found: add the dataset with the bundle zips as Input"

In [ ]:
env = {**os.environ, "HF_HOME": HF_HOME, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
out_root = Path("/kaggle/working/out")
procs = []
for b in sorted(bundles, key=lambda b: json.loads((b / "manifest.json").read_text())["stage"] != "music"):  # music first: it's quick
    stage = json.loads((b / "manifest.json").read_text())["stage"]
    gpu = 1 if (stage == "music" and n > 1) else 0
    cmd = [sys.executable, str(b / "runner.py"), "--bundle", str(b), "--out", str(out_root / b.name), "--gpu", str(gpu)]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    log = open(f"/kaggle/working/{b.name}.log", "w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(b))
    procs.append((b.name, p))
    print("started", b.name, "on GPU", gpu)
    p.wait()            # one after the other: 29 GB of host RAM, not the GPUs, is the limit (FAILURES.md F8)
for name, p in procs:
    p.wait()
    print(name, "finished with exit code", p.returncode)

In [ ]:
# Zip first, so a printing problem can never cost the results.
shutil.make_archive("/kaggle/working/heritage-gpu-out", "zip", out_root)
for log in sorted(Path("/kaggle/working").glob("*.log")):
    print(f"==== {log.name}"); print("".join(log.read_text().splitlines(keepends=True)[-15:]))
for f in sorted(out_root.rglob("results*.jsonl")):
    for line in f.read_text().splitlines():
        r = json.loads(line)
        if r.get("status") == "started":
            continue
        print(r["id"], r["status"], f"{r.get('seconds')}s", f"load {r.get('load_s', '-')}s",
              f"peak {r.get('peak_vram_gb')} GB", r.get("error", "") or "")
print("Download heritage-gpu-out.zip from the Output tab, then: uv run film ingest heritage-gpu-out.zip")
